# Composite Score Test: rank(P) + rank(U)

Quick check — does an additive rank-based composite fix the destructive interference from `(2P-1)*U`?

In [5]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..') / 'src'))
from krauss.backtest.ranking import rank_and_select
from krauss.backtest.portfolio import build_daily_portfolios, aggregate_portfolio_returns
from krauss.backtest.costs import compute_turnover, apply_transaction_costs

ROOT = Path('..')
PROCESSED = ROOT / 'data' / 'processed'

pred2 = pd.read_parquet(PROCESSED / 'predictions_phase2.parquet')
pred1 = pd.read_parquet(PROCESSED / 'predictions_phase1.parquet')
returns = pd.read_parquet(PROCESSED / 'daily_returns.parquet')

pred2['date'] = pd.to_datetime(pred2['date'])
pred1['date'] = pd.to_datetime(pred1['date'])
returns['date'] = pd.to_datetime(returns['date'])

# Compute average rank composite: 0.5*rank(P) + 0.5*rank(U)
for family in ['dnn', 'xgb', 'rf', 'ens1']:
    p_col = f'p_{family}'
    u_col = f'u_{family}'
    pred2[f'rankcomp_{family}'] = 0.5 * (
        pred2.groupby('date')[p_col].rank(method='average', ascending=True) +
        pred2.groupby('date')[u_col].rank(method='average', ascending=True)
    )

print('Average rank composites computed.')
print(f'ENS1 rankcomp: mean={pred2["rankcomp_ens1"].mean():.1f}, '
      f'std={pred2["rankcomp_ens1"].std():.1f}')

Average rank composites computed.
ENS1 rankcomp: mean=248.5, std=126.6


In [7]:
def run_backtest(predictions, score_col, k, returns_df, cost_bps=5):
    sel = rank_and_select(predictions, k=k, score_col=score_col)
    hold = build_daily_portfolios(sel, returns_df, k=k)
    daily = aggregate_portfolio_returns(hold)
    turn = compute_turnover(hold, k=k)
    daily = apply_transaction_costs(daily, turn, cost_bps)
    daily['date'] = pd.to_datetime(daily['date'])
    return daily

K_VALUES = [10, 50]

# Run backtests: P1 baseline, P2 U-only, old composite, new avg rank composite
results = []

for family, p1_col in [('dnn', 'p_dnn'), ('xgb', 'p_xgb'), ('rf', 'p_rf'), ('ens1', 'p_ens1')]:
    m_name = family.upper()
    for k in K_VALUES:
        # P1 baseline
        d = run_backtest(pred1, p1_col, k, returns)
        results.append({'Model': m_name, 'Score': 'P1 Base', 'k': k,
                        'ret/day': d['port_ret'].mean(), 'ret_net/day': d['port_ret_net'].mean()})
        # P2 U-only
        d = run_backtest(pred2, f'score_u_{family}', k, returns)
        results.append({'Model': m_name, 'Score': 'P2 U', 'k': k,
                        'ret/day': d['port_ret'].mean(), 'ret_net/day': d['port_ret_net'].mean()})
        # Old composite (2P-1)*U
        d = run_backtest(pred2, f'score_comp_{family}', k, returns)
        results.append({'Model': m_name, 'Score': 'P2 (2P-1)*U', 'k': k,
                        'ret/day': d['port_ret'].mean(), 'ret_net/day': d['port_ret_net'].mean()})
        # New average rank composite
        d = run_backtest(pred2, f'rankcomp_{family}', k, returns)
        results.append({'Model': m_name, 'Score': 'P2 0.5r(P)+0.5r(U)', 'k': k,
                        'ret/day': d['port_ret'].mean(), 'ret_net/day': d['port_ret_net'].mean()})

res = pd.DataFrame(results)
print('Done.')

Done.


In [8]:
# Focus on k=10 headline
k10 = res[res['k'] == 10].pivot(index='Model', columns='Score', values='ret_net/day')
k10 = k10[['P1 Base', 'P2 U', 'P2 (2P-1)*U', 'P2 0.5r(P)+0.5r(U)']]
k10 = (k10 * 100).round(4)
k10.columns.name = 'Daily net return (%)'
display(k10.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=1)
        .set_caption('k=10 post-cost daily return (%)'))

print()

# Full k grid for ENS1
ens1 = res[res['Model'] == 'ENS1'].pivot(index='Score', columns='k', values='ret_net/day')
ens1 = ens1.reindex(['P1 Base', 'P2 U', 'P2 (2P-1)*U', 'P2 0.5r(P)+0.5r(U)'])
ens1 = (ens1 * 100).round(4)
display(ens1.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=None)
        .set_caption('ENS1 post-cost daily return (%) across k'))

Daily net return (%),P1 Base,P2 U,P2 (2P-1)*U,P2 0.5r(P)+0.5r(U)
Model,,,,
DNN,0.1370,0.0822,0.0937,0.1059
ENS1,0.2788,0.2567,-0.0489,0.2723
RF,0.2709,0.2200,-0.0253,0.2723
XGB,0.2149,0.1835,-0.1244,0.2301


k,10,50
Score,,
P1 Base,0.2788,0.1208
P2 U,0.2567,0.1012
P2 (2P-1)*U,-0.0489,-0.0616
P2 0.5r(P)+0.5r(U),0.2723,0.1208
